# **Lower Bound using Minimum Spanning Tree**

In [ ]:
from PIL import Image
from pathlib import Path
from itertools import *
from functools import *
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

N = 257
NN = N*N
radius = 128

df_image = pd.read_csv('../input/santa-2022/image.csv')

def df_to_image(df):
    side = int(len(df) ** 0.5)  # assumes a square image
    return df_image.set_index(['x', 'y']).to_numpy().reshape(side, side, -1)

image = df_to_image(df_image)

In [ ]:
from scipy.sparse import dok_matrix
import tqdm

def hash_to_xy(i):
    return i//N, i%N
def xy_to_hash(x,y):
    return x*N+y
def get_cost(i, j):
    ix, iy = hash_to_xy(i)
    jx, jy = hash_to_xy(j)
    d = np.abs(ix-jx)+np.abs(iy-jy)
    return np.sum(np.abs(image[ix,iy]-image[jx,jy]))*3 + np.sqrt(d)

mat = dok_matrix((NN, NN), dtype=np.float64)

for i in tqdm.trange(NN):
    x, y = hash_to_xy(i)
    for dx in range(-8, 9):
        for dy in range(-8, 9):
            nx, ny = x+dx, y+dy
            if np.abs(dx)+np.abs(dy) > 8:
                continue
            if nx<0 or N<=nx or ny<0 or N<=ny:
                continue
            j = xy_to_hash(nx, ny)
            cost = get_cost(i, j)
            mat[i,j] = cost

In [ ]:
from scipy.sparse.csgraph import minimum_spanning_tree
mst = minimum_spanning_tree(mat)

In [ ]:
edges = list(zip(*mst.nonzero()))

lines = []
for i, j in edges:
    ix, iy = hash_to_xy(i)
    jx, jy = hash_to_xy(j)
    d = np.sqrt(np.abs(ix-jx)**2 + np.abs(iy-jy)**2)
    # if d >= 1.8:
    lines.append([[iy - radius, radius - ix],[jy - radius, radius - jx]])
    
import matplotlib.collections as mc
lc = mc.LineCollection(lines, colors='b')

fig = plt.figure(figsize=(20,20))
ax = fig.add_subplot(111)
ax.add_collection(lc)

ax.matshow(image, extent=(-radius-0.5, radius+0.5, -radius-0.5, radius+0.5))
ax.grid(None)

ax.autoscale()
fig.show()


In [ ]:
print(f"total cost : {mst.sum()}")